# AgriStress · Earth Engine multi-satellite pipeline (guided tour)

**ISRO BAH 2026 · Problem Statement 6.**

> ⚠️ **This notebook requires Google Earth Engine authentication.**
> Run `earthengine authenticate` (or set a service account) first — see
> `gee/README.md`. Cells that contact EE are clearly marked **[needs EE]** and
> will print a setup recipe instead of crashing if EE is unavailable.

It walks through the `gee/` scripts that fuse **Sentinel-2 + Landsat 8/9**
(optical), **Sentinel-1** (SAR), **SMAP** (soil moisture), **IMERG/CHIRPS**
(rainfall), **ERA5-Land** (ET0 forcing) and the **AlphaEarth** 64-D satellite
embedding into a crop-type map, a moisture-stress view and an 8-day irrigation
advisory, then shows them on `geemap` interactive maps.

The fully offline (no-credentials) demo is `01_quickstart_demo.ipynb`.

## 0 · Initialise Earth Engine  **[needs EE]**
Uses the shared helper `gee/00_auth.py` (re-exported via `gee._auth`). The repo
root is added to `sys.path` so the `gee` package imports cleanly from `notebooks/`.

In [ ]:
import os
import sys
from importlib import import_module

# Make the repo root importable (so `import gee.*` works from notebooks/).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

auth = import_module('gee._auth')

EE_READY = False
try:
    ee = auth.init_ee(project=os.environ.get('EE_PROJECT'))
    EE_READY = True
    print('Earth Engine ready.')
except auth.EarthEngineUnavailable as exc:
    print('Earth Engine NOT initialised — the [needs EE] cells below will be skipped.\n')
    print(exc)

## 1 · Define the pilot AOI & season
A small canal-command-area box (kept tiny so demo `getInfo`/tile calls are fast).
Swap in your own command-area boundary for an operational run.

In [ ]:
# [W, S, E, N] lon/lat — a Bhakra/Patiala-style command-area demo box.
AOI = [76.30, 30.60, 76.55, 30.80]
START, END = '2023-06-01', '2023-11-30'   # kharif season
EMB_YEAR = 2022                            # AlphaEarth annual embedding year

# Load the numbered gee/ modules (names start with a digit → load by file path).
import importlib.util

def load_step(fname, name):
    path = os.path.join(REPO_ROOT, 'gee', fname)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

opt = load_step('01_optical_harmonize.py', 'gee_01')
sar = load_step('02_sar_s1.py', 'gee_02')
idx = load_step('03_indices_phenology.py', 'gee_03')
sm = load_step('04_soil_moisture.py', 'gee_04')
clf = load_step('05_crop_classification.py', 'gee_05')
adv = load_step('06_et0_advisory.py', 'gee_06')
print('Loaded gee steps 01–06. AlphaEarth asset:', clf.SATELLITE_EMBEDDING_COLLECTION)

## 2 · Optical harmonisation — Sentinel-2 + Landsat 8/9  **[needs EE]**
HLS-style: Cloud Score+ `cs_cdf ≥ 0.6` (S2) + `QA_PIXEL` (Landsat) masking,
S2→OLI bandpass adjustment, **8-day median composites**.

In [ ]:
if EE_READY:
    optical = opt.build_harmonized_collection(ee, AOI, START, END)
    optical_8d = opt.to_8day_composites(ee, optical, START, END)
    print('8-day harmonised optical composites:', optical_8d.size().getInfo())
    print('common bands:', opt.COMMON_BANDS)
else:
    print('skipped — Earth Engine not initialised.')

## 3 · SAR — Sentinel-1 GRD  **[needs EE]**
Edge-noise masking, speckle multi-look, `VV`/`VH`/`VH⁄VV`/**RVI**, ASC/DESC-aware
8-day composites — the all-weather witness for monsoon/kharif cloud gaps.

In [ ]:
if EE_READY:
    s1 = sar.load_sentinel1(ee, AOI, START, END)
    s1_8d = sar.to_8day_composites(ee, s1, START, END, orbit_aware=True)
    print('Sentinel-1 8-day composites (ASC+DESC):', s1_8d.size().getInfo())
    print('SAR features:', sar.SAR_BANDS)
else:
    print('skipped — Earth Engine not initialised.')

## 4 · Indices & phenology  **[needs EE]**
NDVI/EVI/NDWI/NDMI/STR time-series + harmonic/threshold SOS/EOS/LGP.

In [ ]:
if EE_READY:
    index_ts = idx.index_time_series(ee, optical_8d)
    phenology = idx.sos_eos_from_threshold(ee, index_ts)
    print('index bands per step:', idx.INDEX_BANDS)
    print('phenology bands     : SOS_doy, EOS_doy, LGP')
else:
    print('skipped — Earth Engine not initialised.')

## 5 · Soil moisture, rainfall & ET0 forcing  **[needs EE]**
SMAP L4 + Sentinel-1 downscaling sketch; IMERG/CHIRPS rainfall; ERA5-Land fields.

In [ ]:
if EE_READY:
    smap = sm.smap_soil_moisture(ee, AOI, START, END)
    rain = sm.chirps_rainfall(ee, AOI, START, END)
    et0_inputs = sm.era5_et0_inputs(ee, AOI, START, END)
    print('SMAP composites:', smap.size().getInfo(), '| CHIRPS:', rain.size().getInfo())
    print('ERA5-Land ET0 fields:', sm.ERA5_ET0_BANDS)
else:
    print('skipped — Earth Engine not initialised.')

## 6 · Crop classification — feature stack **and** AlphaEarth embedding  **[needs EE + labels]**

Two parallel paths, both `smileRandomForest`, both validated with an
`errorMatrix` → **Overall Accuracy + Kappa**:

* **A** — multi-temporal optical-index + Sentinel-1 feature stack;
* **B** — `GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL` 64-D embedding.

Provide `points` as an `ee.FeatureCollection` of labelled samples with an
integer `crop_class` property (ground-truth from the study area).

In [ ]:
if EE_READY:
    embedding = clf.satellite_embedding_image(ee, AOI, EMB_YEAR)
    stack = clf.build_feature_stack(ee, index_ts, sar.to_8day_composites(ee, s1, START, END, orbit_aware=False))
    print('AlphaEarth embedding bands:', len(clf.EMBEDDING_BANDS), '(A00..A63)')

    # --- supply your labelled ground-truth points here ---
    # points = ee.FeatureCollection('users/you/crop_labels')   # needs 'crop_class'
    points = None
    if points is not None:
        res_stack = clf.classify_with_features(ee, stack, points)
        res_emb = clf.classify_with_features(ee, embedding, points)
        print('Path A (stack)     OA =', res_stack['overall_accuracy'].getInfo(),
              'Kappa =', res_stack['kappa'].getInfo())
        print('Path B (AlphaEarth) OA =', res_emb['overall_accuracy'].getInfo(),
              'Kappa =', res_emb['kappa'].getInfo())
        crop_map = res_emb['crop_map']
    else:
        crop_map = None
        print('No labelled points provided — set `points` to train RF and get OA/Kappa.')
else:
    print('skipped — Earth Engine not initialised.')

## 7 · ET0 → ETc → 8-day deficit → irrigation advisory  **[needs EE]**
FAO-56 Penman-Monteith ET0 (ERA5-Land) → Kc-from-NDVI → ETc → rainfall balance →
8-day deficit → advisory classes for the latest 8-day step.

In [ ]:
if EE_READY:
    era5_last = ee.Image(et0_inputs.sort('system:time_start', False).first())
    ndvi_last = ee.Image(index_ts.sort('system:time_start', False).first()).select('NDVI')
    rain_last = ee.Image(rain.sort('system:time_start', False).first()).select('rain_mm')
    advisory = adv.advisory_for_step(ee, era5_last, ndvi_last, rain_last)
    print('advisory bundle bands:', advisory['bundle'].bandNames().getInfo())
    print('classes:', adv.ADVISORY_CLASSES)
else:
    print('skipped — Earth Engine not initialised.')

## 8 · Interactive maps with `geemap`  **[needs EE + geemap]**
Layer the fused products on a slippy map. `geemap` is optional; the cell degrades
gracefully if it is not installed.

In [ ]:
if EE_READY:
    try:
        import geemap
        Map = geemap.Map(center=[30.70, 76.42], zoom=11)
        # True-colour median optical composite.
        rgb = optical_8d.median().select(['red', 'green', 'blue'])
        Map.addLayer(rgb, {'min': 0.0, 'max': 0.3}, 'Optical RGB (8-day median)')
        # NDVI of the latest step.
        Map.addLayer(ndvi_last, {'min': 0, 'max': 1, 'palette': ['white', 'green']}, 'NDVI (latest)')
        # SAR RVI (vegetation density proxy).
        Map.addLayer(s1_8d.select('RVI').median(), {'min': 0, 'max': 1, 'palette': ['blue', 'yellow', 'green']}, 'SAR RVI')
        # 8-day deficit & advisory.
        Map.addLayer(advisory['deficit'], {'min': 0, 'max': 40, 'palette': ['white', 'orange', 'red']}, 'Water deficit (mm/8d)')
        Map.addLayer(advisory['advisory'], {'min': 0, 'max': 3, 'palette': ['#2c7bb6', '#abd9e9', '#fdae61', '#d7191c']}, 'Irrigation advisory')
        if crop_map is not None:
            Map.addLayer(crop_map, {'min': 0, 'max': 9, 'palette': ['#1b9e77', '#d95f02', '#7570b3', '#e7298a']}, 'Crop-type map')
        Map.addLayerControl()
        display(Map)
    except ImportError:
        print('geemap not installed — pip install geemap to view interactive maps.')
else:
    print('skipped — Earth Engine not initialised.')

## 9 · Export gold artefacts (the O(1) factory boundary)  **[needs EE]**
Batch-export the crop map and advisory to EE assets / COGs — read instantly by
the tile server (see `gee/README.md`, *precompute-then-serve*). Tasks are created
here; call `task.start()` to launch.

In [ ]:
if EE_READY:
    ASSET_PREFIX = os.environ.get('EE_ASSET_PREFIX', 'projects/your-gcp-project/assets/agristress')
    tasks = []
    if crop_map is not None:
        tasks.append(clf.export_crop_map(ee, crop_map, f'{ASSET_PREFIX}/crop_map', AOI))
    tasks.append(adv.export_advisory(ee, advisory['bundle'], f'{ASSET_PREFIX}/advisory', AOI))
    print(f'Prepared {len(tasks)} export task(s). Call task.start() on each to launch:')
    for tsk in tasks:
        print('  -', tsk.config.get('description'))
        # tsk.start()   # uncomment to actually run the batch export
else:
    print('skipped — Earth Engine not initialised.')

## Recap
This tour ran the `gee/` pipeline end-to-end: harmonised optical (S2+L8/9) and
SAR (S1), derived indices + phenology, assembled soil-moisture/rainfall/ET0
forcing, classified crops via **both** a multi-temporal feature stack and the
**AlphaEarth 64-D embedding** (with `errorMatrix` OA + Kappa), produced an 8-day
FAO-56 irrigation advisory, and exported gold artefacts for the O(1) tile server.